# V2 CMS 데이터 정합성 검증

## tl;dr

- 2026-08-05 기준 USA와 PL 모두 필수 컬럼과 13개 완료 주는 충족하지만, 음수 매출 때문에 현행 V2 원천 빌드가 차단된다.
- USA는 V1 Biz Type 기준에서 제외되는 판매수량이 4.15%이며, 무료샘플이 정상 수요로 포함되는 것이 가장 큰 모집단 불일치다.
- EU 리드타임 API는 충분한 RAIL·SEA 표본을 반환하지만 현행 V2 서비스는 USA만 API 값을 동적으로 적용한다.
- 단가는 USA 71개, PL 12개 SKU/행이 0이며, 판매단가로 대체하지 않고 확인 필요로 처리해야 한다.

## Context & Methods

검증 대상은 발주분석 V2의 CMS 6개 원천과 법인별 리드타임 API다. 판매 분석기간은 현재 주를 제외한 13개 완료 주(2026-05-04~2026-08-02)이며, 판매 모집단 비교는 현행 V2 전체 행과 V1 법인별 Biz Type 필터를 사용한다. 원천 payload와 SKU 목록은 저장하지 않고 집계 결과만 출력한다.

### Key Assumptions

- `prod_cd`는 원천 간 SKU 연결키다.
- 완전히 동일한 JSON 행은 중복 후보이지 확정 중복은 아니다. 원전표 라인키 확인 전 자동 삭제하지 않는다.
- 음수 매출과 양수 수량의 조합은 반품·취소 후보지만 거래 연결키 없이는 수량 부호를 임의 변경하지 않는다.
- 리드타임은 입고완료일 기준 최근 6개월, Packing No 단위, 1~180일 정수 표본, STDEV.S를 사용한다.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from docs.data_quality.v2_cms_data_quality import run_checks

AS_OF = '2026-08-05'

## Data

### 1. Load and validate the bounded CMS snapshot

In [2]:
results = run_checks(AS_OF)
pd.DataFrame([results['summary']])

[perf][cms_fetch_cache] hit source=disk as_of=2026-08-05 key=d0cbb8321009 age_seconds=10882.27 ttl_seconds=14400


[perf][cms_api] request_done path=/us/logistics/lead-time status=200 attempt=1 seconds=0.654 rows=768


[perf][cms_api] paged_done path=/us/logistics/lead-time pages=1 page_size=2000 rows=768 seconds=0.656


[perf][cms_fetch_cache] hit source=disk as_of=2026-08-05 key=4056b9b6c207 age_seconds=154.041 ttl_seconds=14400


[perf][cms_api] request_done path=/eu/logistics/lead-time status=200 attempt=1 seconds=1.456 rows=1092


[perf][cms_api] paged_done path=/eu/logistics/lead-time pages=1 page_size=2000 rows=1092 seconds=1.458


,critical,high,medium,low,entities_ready,entities_checked
0,2,5,11,1,0,2


## Results

### 2. Source completeness and duplicate candidates

In [3]:
endpoint_rows = []
for entity, profile in results['entities'].items():
    for row in profile['endpoints']:
        endpoint_rows.append({
            'entity': entity,
            'endpoint': row['endpoint'],
            'rows': row['rows'],
            'required_missing_rows': row['rows_with_any_required_missing'],
            'exact_duplicate_extra_rows': row['exact_duplicate_extra_rows'],
        })
endpoint_df = pd.DataFrame(endpoint_rows)
display(endpoint_df)

,entity,endpoint,rows,required_missing_rows,exact_duplicate_extra_rows
0,USA,stock_local,16079,0,0
1,USA,stock_hq,10406,0,0
2,USA,sales_local,115310,0,123
3,USA,sales_hq,5430,0,35
4,USA,shipping,1903,0,19
5,USA,open_po,1846,0,0
6,PL,stock_local,2598,0,0
7,PL,stock_hq,3047,0,0
8,PL,sales_local,71761,0,363
9,PL,sales_hq,3323,0,12


### 3. Sales population, negative transactions, and unit cost

In [4]:
sales_rows = []
for entity, profile in results['entities'].items():
    sales = profile['sales']
    price = profile['local_stock']['unit_cost']
    sales_rows.append({
        'entity': entity,
        'period_rows': sales['period_rows'],
        'v1_excluded_rows': sales['v1_excluded_rows'],
        'v1_excluded_qty_pct': sales['v1_excluded_qty_pct'],
        'eligible_negative_amount_rows': sales['eligible_negative_amount_rows'],
        'negative_amount_with_positive_qty_rows': sales['negative_amount_with_positive_qty_rows'],
        'zero_or_missing_unit_cost': price['zero'] + price['null_or_invalid'] + price['negative'],
        'v2_status': profile['current_v2_source_status']['status'],
    })
sales_df = pd.DataFrame(sales_rows)
display(sales_df)

,entity,period_rows,v1_excluded_rows,v1_excluded_qty_pct,eligible_negative_amount_rows,negative_amount_with_positive_qty_rows,zero_or_missing_unit_cost,v2_status
0,USA,112403,4760,4.1487,300,300,71,blocked
1,PL,69688,634,0.1071,161,161,12,blocked


### 4. Cross-source SKU coverage and lead-time samples

In [5]:
coverage_rows = []
lead_rows = []
for entity, profile in results['entities'].items():
    for check, value in profile['integrity'].items():
        coverage_rows.append({'entity': entity, 'check': check, **value})
    lead = profile['lead_time']
    if lead.get('api_status') == 'ready':
        for mode, value in lead['modes'].items():
            lead_rows.append({'entity': entity, 'mode': mode, **value})
display(pd.DataFrame(coverage_rows))
display(pd.DataFrame(lead_rows))

,entity,check,child_skus,matched_skus,missing_skus,match_pct
0,USA,sales_to_local_stock,4684,4684,0,100.0000
1,USA,sales_to_any_stock,4684,4684,0,100.0000
2,USA,shipping_to_any_stock,1185,1161,24,97.9747
3,USA,open_po_to_any_stock,1846,1764,82,95.5580
4,PL,sales_to_local_stock,1990,1989,1,99.9497
5,PL,sales_to_any_stock,1990,1989,1,99.9497
6,PL,shipping_to_any_stock,837,836,1,99.8805
7,PL,open_po_to_any_stock,1302,1090,212,83.7174


,entity,mode,sample_size,mean_days,stdev_days,sigma_weeks
0,USA,AIR,125,6.1520,9.1828,1.3118
1,USA,SEA,121,29.1736,5.9395,0.8485
2,USA,RAIL,0,NaN,NaN,NaN
3,USA,TRUCK,0,NaN,NaN,NaN
4,PL,AIR,218,14.5092,6.0628,0.8661
5,PL,SEA,178,72.7360,14.0814,2.0116
6,PL,RAIL,72,46.2222,11.2052,1.6007
7,PL,TRUCK,5,30.4000,2.1909,0.3130


### 5. Prioritized quality findings

In [6]:
issues_df = pd.DataFrame(results['issues'])
display(issues_df)

,entity,severity,check,evidence,risk,action
0,PL,Critical,V2 source build,판매 원천의 매출 값이 음수입니다: ALSS03-S30EU,해당 법인의 V2 전체 계산이 중단됨,Biz Type 필터 후 음수 거래 정규화 규칙 적용
1,USA,Critical,V2 source build,판매 원천의 매출 값이 음수입니다: KoeP11-MFM,해당 법인의 V2 전체 계산이 중단됨,Biz Type 필터 후 음수 거래 정규화 규칙 적용
2,PL,High,EU lead-time integration,EU 리드타임 API는 조회되지만 현행 V2 서비스는 USA만 동적 반영,EU가 최신 API 표본 대신 고정 리드타임·표준편차를 사용,EU RAIL·SEA 표본 집계를 V2 설정과 감사정보에 연결
3,PL,High,Negative open-PO quantity,음수 미입고수량 142행,해당 SKU는 V2 입력 검증에서 계산불가 처리됨,취소·감액 전표 여부를 구분하고 원천 open_qty 의미를 확정
4,PL,High,Open-PO-to-stock coverage,재고마스터 미연결 212 SKU (16.28%),수량은 존재하지만 재고·상품·단가 정보가 불완전함,법인 재고마스터와 상품코드 등록 상태를 대조
5,USA,High,Biz Type consistency,"V1 제외 4,760행, 수량 비중 4.15%",V1과 V2 수요 모집단 불일치,V2에서 공통 filter_sales_by_biz_type 재사용
6,USA,High,Negative open-PO quantity,음수 미입고수량 65행,해당 SKU는 V2 입력 검증에서 계산불가 처리됨,취소·감액 전표 여부를 구분하고 원천 open_qty 의미를 확정
7,PL,Medium,Biz Type consistency,"V1 제외 634행, 수량 비중 0.11%",V1과 V2 수요 모집단 불일치,V2에서 공통 filter_sales_by_biz_type 재사용
8,PL,Medium,Exact duplicate local sales rows,완전 동일 중복 초과 행 363건,거래 행이 실제 중복이면 주간 수요와 순매출이 이중 집계됨,Invoice/원전표/라인 식별자로 정상 복수행과 중복 적재를 구분
9,PL,Medium,Local unit cost completeness,양수 stock_ucost 미확보 12 SKU/행,발주수량은 계산돼도 제안금액이 0 또는 미계산될 수 있음,0·누락 단가를 확인 필요로 표시하고 판매단가 fallback 금지


## Takeaways

1. 현재 스냅샷은 필수값·기간·통화·판매-재고 연결률은 대체로 양호하다.
2. 운영 차단 원인은 결측이 아니라 음수 매출을 전역 오류로 처리하는 변환 규칙이다.
3. V2는 V1 Biz Type 필터를 먼저 적용하고, 허용된 모집단 안에서 반품·취소를 별도 정규화해야 한다.
4. EU API에는 RAIL·SEA 표본이 충분하므로 고정값을 유지할 근거가 없다.
5. 음수 미입고수량, 재고마스터 미연결, 완전 동일 중복 후보는 자동 보정하지 말고 검증 큐와 감사 지표로 관리해야 한다.